In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Execute lakebase_schema.sql (control, ct_checkpoint, reseed_queue) idempotently.
# Strip -- comments first (they may contain ';'), then split into statements. Safe for this
# controlled DDL (no ';' or '--' inside string literals).
import re

schema_path = "/Workspace" + nb_dir + "/lakebase_schema.sql"
with open(schema_path) as f:
    ddl = f.read()
no_comments = "\n".join(re.sub(r"--.*$", "", line) for line in ddl.splitlines())
statements = [s.strip() for s in no_comments.split(";") if s.strip()]

with store.connection.cursor() as cur:
    for stmt in statements:
        cur.execute(stmt)
print(f"applied {len(statements)} DDL statements to {db_name}")